# 🌿 Space Journey: London Sensory Impact Score (SIS v1.1) & Copernicus DEM Explorer
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/spacelabdevteam-hub/sensory-impact-score-datasets/blob/main/cities/03_london/explore_london_sensory_impact.ipynb)
[![Hugging Face Dataset](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-london--sensory--impact--dataset-FFD21E)](https://huggingface.co/datasets/spacejourney/london-sensory-impact-dataset)
[![DOI](https://zenodo.org/badge/DOI/10.5281/zenodo.22756473.svg)](https://doi.org/10.5281/zenodo.22756473)
[![WHO Acoustic Guideline](https://img.shields.io/badge/WHO_Guideline-53_dB_Lden-006699)](https://spacejourney.app/standards/sensory-impact-score/)
[![Live MCP Server](https://img.shields.io/badge/MCP_Endpoint-mcp.spacelab.network-10B981)](https://mcp.spacelab.network/mcp)

Interactive notebook for the **London, UK Sensory Impact Score (SIS v1.1)** benchmark curated by [Space Journey](https://spacejourney.app/standards/sensory-impact-score/), combining **World Health Organization (WHO) 53 dB Lden** acoustic alignment with **ESA Copernicus DEM GLO-90** satellite elevation and biomechanical exertion telemetry across 15 contemplative sanctuaries.

Running this notebook in Google Colab downloads the canonical CSV benchmark directly from the Hugging Face Hub (with automatic fallback to GitHub raw).

In [1]:
# 1. Install dependencies
!pip install -q datasets huggingface_hub pandas matplotlib seaborn

In [2]:
from datasets import load_dataset
import pandas as pd

GITHUB_RAW = 'https://raw.githubusercontent.com/spacelabdevteam-hub/sensory-impact-score-datasets/main/cities/03_london/data/london_sensory_impact_sis_v1.csv'

try:
    # Explicitly target the canonical flat CSV file on Hugging Face Hub
    ds = load_dataset('spacejourney/london-sensory-impact-dataset', data_files='data/london_sensory_impact_sis_v1.csv')
    df = pd.DataFrame(ds['train'])
    source_label = 'spacejourney/london-sensory-impact-dataset (data/london_sensory_impact_sis_v1.csv)'
except Exception:
    df = pd.read_csv(GITHUB_RAW)
    source_label = 'GitHub raw mirror (cities/03_london/data/london_sensory_impact_sis_v1.csv)'

who_ok = int((df['acoustic_noise_db'] <= 53).sum())
print(f'✅ Successfully loaded {len(df)} contemplative sanctuaries from {source_label}')
print(f"   Mean SIS: {df['sis_score'].mean():.1f}/100 | Mean Acoustic Level: {df['acoustic_noise_db'].mean():.1f} dB | WHO <= 53 dB Lden: {who_ok}/{len(df)} sanctuaries")
cols = ['id', 'name', 'category', 'sis_score', 'acoustic_noise_db', 'elevation_m', 'biomechanical_exertion_index', 'golden_hour_window']
df[cols].head(10)

✅ Successfully loaded 15 contemplative sanctuaries from spacejourney/london-sensory-impact-dataset (data/london_sensory_impact_sis_v1.csv)
   Mean SIS: 92.3/100 | Mean Acoustic Level: 43.9 dB | WHO <= 53 dB Lden: 13/15 sanctuaries

  id | name                                     | category         | sis | dB | elev_m | exert | golden_hour_window
-----+------------------------------------------+------------------+-----+----+--------+-------+----------------------
L-01 | Sky Garden | The forest in the clouds    | Free Restorative |  90 | 45 |   40.0 |  31.5 | 17:30 - 19:00 GMT/BST
L-02 | Primrose Hill | London skyline           | Free Restorative |  97 | 25 |   55.0 |  47.7 | 06:00 - 07:30 GMT/BST
L-03 | St Dunstan in the East: The Garden Among | Free Restorative |  93 | 45 |   26.0 |  20.3 | 12:00 - 14:00 GMT/BST
L-04 | Little Venice: The quietest walk through | Free Restorative |  93 | 45 |   31.0 |  28.5 | 08:30 - 10:00 GMT/BST
L-05 | Kyoto Garden: The Japanese garden in Lon | Free Re

In [3]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6), dpi=120)
sns.set_theme(style='darkgrid')

# Verified Solarpunk palette matching exact CSV category values
palette_map = {'Curated Premium': '#10B981', 'Free Restorative': '#E8C57D', 'Secret Sanctuary': '#34D399'}
sns.scatterplot(
    data=df,
    x='acoustic_noise_db',
    y='sis_score',
    hue='category',
    palette=palette_map,
    s=160,
    edgecolor='#064E3B',
    linewidth=1.2
)
plt.title('London, UK: Acoustic Pressure (dB) vs. Sensory Impact Score (SIS)', fontsize=13, weight='bold', pad=14)
plt.xlabel('Acoustic Level (dB — Lower = Quieter Sanctuary)', fontsize=11)
plt.ylabel('Sensory Impact Score (SIS 0–100)', fontsize=11)
plt.axvline(x=53, color='#059669', linestyle='--', linewidth=1.5, label='WHO Restorative Guideline (53 dB Lden)')

for _, row in df.sort_values(['sis_score', 'acoustic_noise_db'], ascending=[False, True]).head(6).iterrows():
    short_name = str(row['name']).split(':')[0].split('|')[0].strip()[:26]
    plt.annotate(
        short_name, (row['acoustic_noise_db'], row['sis_score']),
        textcoords='offset points', xytext=(0, 9), ha='center', fontsize=8, weight='semibold'
    )

plt.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [4]:
# ESA Copernicus DEM GLO-90: Topographic Elevation vs. Biomechanical Exertion Index
plt.figure(figsize=(10, 5.5), dpi=120)
sns.set_theme(style='darkgrid')

ordered = df.sort_values('elevation_m', ascending=True)
colors = ['#10B981' if x < 25 else '#34D399' if x < 36 else '#E8C57D' for x in ordered['biomechanical_exertion_index']]
labels = [f"{r['id']} · {str(r['name']).split(':')[0].split('|')[0].strip()[:22]}" for _, r in ordered.iterrows()]

plt.barh(labels, ordered['elevation_m'], color=colors, edgecolor='#064E3B', linewidth=0.9)
plt.axvline(x=11.0, color='#059669', linestyle='--', linewidth=1.4, label='London Thames Basin Baseline (11.0 m)')
plt.title('London Sanctuaries: ESA Copernicus DEM GLO-90 Elevation Profile (m)', fontsize=13, weight='bold', pad=12)
plt.xlabel('Satellite Elevation Above Sea Level (m)', fontsize=11)
plt.ylabel('Verified Sanctuary Node', fontsize=11)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [5]:
top_5 = df.sort_values(by=['sis_score', 'acoustic_noise_db'], ascending=[False, True]).head(5)

print('🏆 TOP 5 RESTORATIVE SANCTUARIES — LONDON, UK')
print('=' * 75)
for idx, (_, row) in enumerate(top_5.iterrows(), 1):
    print(f"\n{idx}. {row['name']} (SIS: {row['sis_score']}/100 | {row['acoustic_noise_db']} dB | {row['elevation_m']} m DEM)")
    print(f"   Tier: {row['category']} | Neighborhood: {row['neighborhood']}")
    print(f"   Optimal Window: {row['golden_hour_window']} | Duration: {row['duration_minutes']} min")
    print(f"   🔗 Official Sanctuary Guide: {row['canonical_url']}")

print('\n✨ 1-Click Auto-Import Itinerary into Space Journey Planner:')
print('https://spacejourney.app/en/itinerary-planner/?ai_city=londres&ai_days=3&utm_source=colab_notebook')

🏆 TOP 5 RESTORATIVE SANCTUARIES — LONDON, UK

1. Primrose Hill | London skyline (SIS: 97/100 | 25 dB | 55.0 m DEM)
   Tier: Free Restorative | Neighborhood: Primrose Hill (Camden, North London)
   Optimal Window: 06:00 - 07:30 GMT/BST | Duration: 90 min
   🔗 Official Sanctuary Guide: https://spacejourney.app/en/experiencia/primrose-hill-london-skyline/

2. Kyoto Garden: The Japanese garden in London (SIS: 97/100 | 30 dB | 37.0 m DEM)
   Tier: Free Restorative | Neighborhood: Kensington (Holland Park)
   Optimal Window: 08:00 - 09:30 GMT/BST | Duration: 90 min
   🔗 Official Sanctuary Guide: https://spacejourney.app/en/experiencia/kyoto-garden-the-japanese-garden-in-london/

3. Barbican Conservatory: London's solarpunk oasis (SIS: 97/100 | 40 dB | 30.0 m DEM)
   Tier: Secret Sanctuary | Neighborhood: City of London (Barbican Estate)
   Optimal Window: 10:00 - 11:00 GMT/BST | Duration: 60 min
   🔗 Official Sanctuary Guide: https://spacejourney.app/en/experiencia/barbican-conservatory-lond